# PortPy Tutorial: Full Function Reference

A working debug notebook that exercises **every public function** in every `portpy.metrics`
submodule, plus the `portpy.core` and `portpy.explain` helpers.

**Data**: real market data pulled live from **Alpaca** (stocks, ETFs, commodity ETFs, crypto)
and a risk-free rate pulled live from the **Federal Reserve (FRED)**.

**Portfolios**: three independent portfolios are built, each spanning all five asset
categories (stocks, ETFs, crypto, commodities, a Fed-rate-derived cash leg) and each
carrying at least one **negative (short) weight**, so every metric below is exercised on
long-only and long/short books:

| Portfolio | Style | Shorts |
|---|---|---|
| `growth` | US growth stocks + crypto | short `XOM` (energy hedge) |
| `balanced` | Broad multi-asset diversification | short `NVDA` (valuation hedge) |
| `macro` | Crypto-heavy long/short macro | short `USO`, short `SPY` |

Sections below map 1:1 to PortPy's `metrics/` submodules: returns, risk, performance,
drawdowns, rolling, distributions, benchmarks, regressions, covariance, summary, costs -
followed by the explainability layer and the `Portfolio.metrics` auto-fill mechanics.

`portpy.models`, `portpy.strategies`, and `portpy.visualization` are not yet implemented in
this PortPy version (0.1.0), so they're out of scope here.


## 0. Setup


In [144]:
from __future__ import annotations

import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from alpaca.data.historical import CryptoHistoricalDataClient, StockHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest, StockBarsRequest
from alpaca.data.timeframe import TimeFrame

import portpy
from portpy import Portfolio
from portpy.core import (
    AssetClass,
    align_calendars,
    calendar_coverage_report,
    convert_to_base_currency,
    detect_frequency,
    equal_weights,
    normalize_weights,
)
from portpy.explain import available, get

load_dotenv()
ALPACA_KEY = os.getenv("ALPACA_KEY")
ALPACA_SECRET = os.getenv("ALPACA_SECRET")
assert ALPACA_KEY and ALPACA_SECRET, "Set ALPACA_KEY / ALPACA_SECRET in a .env file."
print("PortPy version:", portpy.__version__)
print("Alpaca credentials loaded:", bool(ALPACA_KEY) and bool(ALPACA_SECRET))


PortPy version: 0.1.0
Alpaca credentials loaded: True


## 1. Data Acquisition -- Alpaca (stocks, ETFs, commodities, crypto) + FRED (risk-free rate)

Universe:
- **Stocks**: AAPL, MSFT, NVDA, JPM, XOM
- **ETFs**: SPY, QQQ, IWM
- **Commodities** (via liquid ETF proxies, fetched through the same Alpaca stock client): GLD (gold), SLV (silver), USO (oil)
- **Crypto** (Alpaca, 24/7): BTC/USD, ETH/USD, SOL/USD
- **Risk-free**: 3-Month Treasury Bill yield (`DGS3MO`) straight from the Fed's FRED database


In [145]:
LOOKBACK_DAYS = 4 * 365

STOCKS = ["AAPL", "MSFT", "NVDA", "JPM", "XOM"]
ETFS = ["SPY", "QQQ", "IWM"]
COMMODITIES = ["GLD", "SLV", "USO"]
CRYPTO = ["BTC/USD", "ETH/USD", "SOL/USD"]

stock_client = StockHistoricalDataClient(ALPACA_KEY, ALPACA_SECRET)
equities_request = StockBarsRequest(
    symbol_or_symbols=STOCKS + ETFS + COMMODITIES,
    timeframe=TimeFrame.Day,
    start=datetime.now() - timedelta(days=LOOKBACK_DAYS),
)
equities_bars = stock_client.get_stock_bars(equities_request)
raw_equities = equities_bars.df["close"].unstack(level="symbol")
raw_equities.index = raw_equities.index.tz_localize(None).normalize()
print(f"Fetched {raw_equities.shape[1]} stock/ETF/commodity symbols, {raw_equities.shape[0]} rows.")
raw_equities.tail()


Fetched 11 stock/ETF/commodity symbols, 1002 rows.


symbol,AAPL,GLD,IWM,JPM,MSFT,NVDA,QQQ,SLV,SPY,USO,XOM
timestamp,,,,,,,,,,,
2026-07-30,333.4300,377.1600,292.5900,350.8500,451.1000,195.0400,683.5500,53.5000,741.6900,127.4800,156.9700
2026-07-31,308.9100,371.5400,291.2000,351.7900,464.7200,200.7500,687.9900,52.3600,747.0300,129.1700,155.4400
2026-08-03,303.4200,371.7100,296.2200,352.6400,487.6500,206.6400,700.0700,52.4600,757.6700,122.1200,155.0600
2026-08-04,309.3800,374.1600,301.7100,357.5200,492.8100,211.9400,723.8500,53.8400,771.3300,115.7800,153.9600
2026-08-05,308.5400,390.8000,300.5900,360.5200,489.8000,221.7300,720.4600,56.2550,771.5500,115.1200,152.1950


In [146]:
crypto_client = CryptoHistoricalDataClient()
crypto_request = CryptoBarsRequest(
    symbol_or_symbols=CRYPTO,
    timeframe=TimeFrame.Day,
    start=datetime.now() - timedelta(days=LOOKBACK_DAYS),
)
crypto_bars = crypto_client.get_crypto_bars(crypto_request)
raw_crypto = crypto_bars.df["close"].unstack(level="symbol")
raw_crypto.index = raw_crypto.index.tz_localize(None).normalize()
raw_crypto.columns = [c.replace("/USD", "") for c in raw_crypto.columns]
print(f"Fetched {raw_crypto.shape[1]} crypto symbols, {raw_crypto.shape[0]} rows.")
raw_crypto.tail()


Fetched 3 crypto symbols, 1460 rows.


,BTC,ETH,SOL
timestamp,,,
2026-08-01,"62,760.2050","1,843.1200",71.8930
2026-08-02,"63,511.3300","1,883.2915",73.5384
2026-08-03,"63,454.2350","1,858.3850",73.4855
2026-08-04,"64,062.7890","1,867.6100",73.7003
2026-08-05,"64,750.1495","1,917.5380",74.5700


In [147]:
# The Fed publishes this on FRED; the plain CSV export needs no API key.
def fetch_fred_series(series_id: str, start: str, end: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}&coed={end}"
    raw = pd.read_csv(url, index_col=0, parse_dates=True)
    return raw[series_id].rename(series_id)

fred_start = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
fred_end = datetime.now().strftime("%Y-%m-%d")

t3m_yield_pct = fetch_fred_series("DGS3MO", fred_start, fred_end)  # 3-Month Treasury, Secondary Market (Fed H.15)
fed_funds_pct = fetch_fred_series("FEDFUNDS", fred_start, fred_end)  # Effective Federal Funds Rate (monthly)

risk_free_annual = (t3m_yield_pct / 100.0).dropna()
print(f"Fetched {len(risk_free_annual)} daily 3-Month T-Bill observations from FRED.")
print(
    f"Latest 3M T-Bill yield: {risk_free_annual.iloc[-1]:.2%} annual  |  "
    f"Latest Fed Funds rate: {fed_funds_pct.dropna().iloc[-1] / 100:.2%} annual"
)
risk_free_annual.tail()


Fetched 996 daily 3-Month T-Bill observations from FRED.
Latest 3M T-Bill yield: 3.91% annual  |  Latest Fed Funds rate: 3.63% annual


observation_date
2026-07-28   0.0390
2026-07-29   0.0383
2026-07-30   0.0382
2026-07-31   0.0383
2026-08-03   0.0391
Name: DGS3MO, dtype: float64

## 2. Calendar Alignment (`portpy.core`) -- combining a 24/7 crypto calendar with Mon-Fri equities

PortPy never aligns data for you. `detect_frequency`, `calendar_coverage_report`, and
`align_calendars` are the tools you call *before* constructing a `Portfolio`.


In [148]:
combined_raw = pd.concat([raw_equities, raw_crypto], axis=1, sort=True)
print(f"detect_frequency: equities -> {detect_frequency(raw_equities.index)!r}, crypto -> {detect_frequency(raw_crypto.index)!r}")
calendar_coverage_report(combined_raw)


detect_frequency: equities -> 'daily-5', crypto -> 'daily-7'


,frequency,first_date,last_date,n_observations,n_missing_in_frame
asset,,,,,
AAPL,daily-5,2022-08-08,2026-08-05,1002,458
GLD,daily-5,2022-08-08,2026-08-05,1002,458
IWM,daily-5,2022-08-08,2026-08-05,1002,458
JPM,daily-5,2022-08-08,2026-08-05,1002,458
MSFT,daily-5,2022-08-08,2026-08-05,1002,458
NVDA,daily-5,2022-08-08,2026-08-05,1002,458
QQQ,daily-5,2022-08-08,2026-08-05,1002,458
SLV,daily-5,2022-08-08,2026-08-05,1002,458
SPY,daily-5,2022-08-08,2026-08-05,1002,458


In [149]:
n_before = len(combined_raw)
aligned_intersection = align_calendars(combined_raw, method="intersection")
aligned_union = align_calendars(combined_raw, method="ffill_union")
aligned_bdays = align_calendars(combined_raw, method="business_days")

print(f"Before alignment:  {n_before} rows, {int(combined_raw.isna().any(axis=1).sum())} rows with >=1 NaN.")
print(f"'intersection'  -> {len(aligned_intersection)} rows (equity-only calendar; crypto weekend moves dropped)")
print(f"'ffill_union'   -> {len(aligned_union)} rows (every day crypto trades; weekend equity prices carried forward)")
print(f"'business_days' -> {len(aligned_bdays)} rows (Mon-Fri only; everything forward-filled onto it)")
assert aligned_union.isna().sum().sum() == 0


Before alignment:  1460 rows, 744 rows with >=1 NaN.
'intersection'  -> 716 rows (equity-only calendar; crypto weekend moves dropped)
'ffill_union'   -> 1459 rows (every day crypto trades; weekend equity prices carried forward)
'business_days' -> 1043 rows (Mon-Fri only; everything forward-filled onto it)


## 3. Building a Synthetic CASH Leg from the Fed's Risk-Free Rate

Every metric function needing an `rf` accepts a plain float, but a *portfolio holding* has to
be a price series. This turns the Fed's daily T-Bill yield into a compounding NAV-style price
column so "risk-free" can sit in the price DataFrame right alongside stocks/ETFs/crypto/commodities.


In [150]:
def synthetic_cash_price(annual_rate: pd.Series, target_index: pd.DatetimeIndex, periods_per_year: int = 365, base: float = 100.0) -> pd.Series:
    rate_on_calendar = annual_rate.reindex(target_index).ffill().bfill()
    daily_rate = (1.0 + rate_on_calendar) ** (1.0 / periods_per_year) - 1.0
    return (base * (1.0 + daily_rate).cumprod()).rename("CASH")

cash_price = synthetic_cash_price(risk_free_annual, aligned_union.index)

master = aligned_union.copy()
master["CASH"] = cash_price
print(f"Master frame: {master.shape[1]} assets x {master.shape[0]} days ({master.index.min().date()} -> {master.index.max().date()})")
master.tail()


Master frame: 15 assets x 1459 days (2022-08-08 -> 2026-08-05)


,AAPL,GLD,IWM,JPM,MSFT,NVDA,QQQ,SLV,SPY,USO,XOM,BTC,ETH,SOL,CASH
timestamp,,,,,,,,,,,,,,,
2026-08-01,308.9100,371.5400,291.2000,351.7900,464.7200,200.7500,687.9900,52.3600,747.0300,129.1700,155.4400,"62,760.2050","1,843.1200",71.8930,119.6291
2026-08-02,308.9100,371.5400,291.2000,351.7900,464.7200,200.7500,687.9900,52.3600,747.0300,129.1700,155.4400,"63,511.3300","1,883.2915",73.5384,119.6414
2026-08-03,303.4200,371.7100,296.2200,352.6400,487.6500,206.6400,700.0700,52.4600,757.6700,122.1200,155.0600,"63,454.2350","1,858.3850",73.4855,119.6539
2026-08-04,309.3800,374.1600,301.7100,357.5200,492.8100,211.9400,723.8500,53.8400,771.3300,115.7800,153.9600,"64,062.7890","1,867.6100",73.7003,119.6665
2026-08-05,308.5400,390.8000,300.5900,360.5200,489.8000,221.7300,720.4600,56.2550,771.5500,115.1200,152.1950,"64,750.1495","1,917.5380",74.5700,119.6791


## 4. Building Three Portfolios (`portpy.core.weights`, `portpy.Portfolio`)

Each portfolio deliberately carries at least one **negative weight** (a short position) to make
sure every metric below is exercised on a long/short book, not just a long-only one.


In [151]:
weights_growth = {
    "AAPL": 0.22, "MSFT": 0.18, "NVDA": 0.18,    # US mega-cap tech (stocks)
    "QQQ": 0.12,                                 # Nasdaq-100 ETF
    "BTC": 0.14, "ETH": 0.08,                    # Crypto
    "GLD": 0.08,                                 # Commodity (gold)
    "CASH": 0.05,                                # Fed risk-free leg
    "XOM": -0.15,                                # SHORT: energy hedge against the growth book
}

weights_balanced = {
    "SPY": 0.20, "IWM": 0.10,                    # Broad + small-cap equity ETFs
    "JPM": 0.12,                                 # Stock (financials)
    "GLD": 0.12, "SLV": 0.08, "USO": 0.08,       # Commodities
    "BTC": 0.10, "ETH": 0.07,                    # Crypto
    "CASH": 0.15,                                # Fed risk-free leg
    "NVDA": -0.10,                               # SHORT: valuation hedge on AI momentum
}

weights_macro = {
    "MSFT": 0.10,                                # Stock anchor
    "QQQ": 0.15,                                 # Long tech ETF
    "BTC": 0.25, "ETH": 0.15, "SOL": 0.10,       # Crypto-heavy book
    "GLD": 0.15,                                 # Long commodity
    "USO": -0.10,                                # SHORT: macro bet against energy
    "SPY": -0.15,                                # SHORT: net market hedge
    "CASH": 0.20,                                # Fed risk-free ballast
}

asset_class_map = {
    "AAPL": AssetClass.EQUITY, "MSFT": AssetClass.EQUITY, "NVDA": AssetClass.EQUITY,
    "JPM": AssetClass.EQUITY, "XOM": AssetClass.EQUITY,
    "SPY": AssetClass.EQUITY, "QQQ": AssetClass.EQUITY, "IWM": AssetClass.EQUITY,
    "GLD": AssetClass.COMMODITY, "SLV": AssetClass.COMMODITY, "USO": AssetClass.COMMODITY,
    "BTC": AssetClass.CRYPTO, "ETH": AssetClass.CRYPTO, "SOL": AssetClass.CRYPTO,
    "CASH": AssetClass.BOND,
}

for label, w in [("growth", weights_growth), ("balanced", weights_balanced), ("macro", weights_macro)]:
    gross = sum(abs(v) for v in w.values())
    net = sum(w.values())
    shorts = {k: v for k, v in w.items() if v < 0}
    print(f"{label:10s}: {len(w)} assets | gross exposure {gross:.2f} | net exposure {net:.2f} | shorts: {shorts}")


growth    : 9 assets | gross exposure 1.20 | net exposure 0.90 | shorts: {'XOM': -0.15}
balanced  : 10 assets | gross exposure 1.12 | net exposure 0.92 | shorts: {'NVDA': -0.1}
macro     : 9 assets | gross exposure 1.35 | net exposure 0.85 | shorts: {'USO': -0.1, 'SPY': -0.15}


### 4.1 `portpy.core.weights` -- `equal_weights` and `normalize_weights` on a long/short book


In [152]:
print("equal_weights on 4 assets:")
print(equal_weights(["A", "B", "C", "D"]))

print("\nnormalize_weights rescales an arbitrary long/short vector to sum to 1 (default allow_negative=True):")
print(normalize_weights(weights_macro))

print("\nnormalize_weights with allow_negative=False raises on a book with shorts (e.g. for a long-only optimizer):")
try:
    normalize_weights(weights_macro, allow_negative=False)
except ValueError as exc:
    print(f"  ValueError: {exc}")


equal_weights on 4 assets:
A   0.2500
B   0.2500
C   0.2500
D   0.2500
Name: weight, dtype: float64

normalize_weights rescales an arbitrary long/short vector to sum to 1 (default allow_negative=True):
MSFT    0.1176
QQQ     0.1765
BTC     0.2941
ETH     0.1765
SOL     0.1176
GLD     0.1765
USO    -0.1176
SPY    -0.1765
CASH    0.2353
Name: weight, dtype: float64

normalize_weights with allow_negative=False raises on a book with shorts (e.g. for a long-only optimizer):
  ValueError: Negative weights are not allowed here (long-only constraint).


In [153]:
def build_portfolio(name: str, weights: dict, frequency: int = 365) -> Portfolio:
    cols = list(weights.keys())
    classes = {k: asset_class_map[k] for k in cols}
    return Portfolio(
        master[cols],
        weights=weights,
        name=name,
        frequency=frequency,
        risk_free_rate=risk_free_annual.iloc[-1],
        asset_classes=classes,
    )

p_growth = build_portfolio("Growth & Crypto Long/Short", weights_growth)
p_balanced = build_portfolio("Diversified Multi-Asset Balanced", weights_balanced)
p_macro = build_portfolio("Long/Short Macro", weights_macro)

portfolios = {"growth": p_growth, "balanced": p_balanced, "macro": p_macro}
for p in portfolios.values():
    print(p)
    print("  weights:", p.weights.round(4).to_dict())
    print("  asset_classes:", p.asset_classes)


Portfolio(name='Growth & Crypto Long/Short', assets=9, n_obs=1459, frequency=365)
  weights: {'AAPL': 0.2444, 'MSFT': 0.2, 'NVDA': 0.2, 'QQQ': 0.1333, 'BTC': 0.1556, 'ETH': 0.0889, 'GLD': 0.0889, 'CASH': 0.0556, 'XOM': -0.1667}
  asset_classes: {'AAPL': <AssetClass.EQUITY: 'equity'>, 'MSFT': <AssetClass.EQUITY: 'equity'>, 'NVDA': <AssetClass.EQUITY: 'equity'>, 'QQQ': <AssetClass.EQUITY: 'equity'>, 'BTC': <AssetClass.CRYPTO: 'crypto'>, 'ETH': <AssetClass.CRYPTO: 'crypto'>, 'GLD': <AssetClass.COMMODITY: 'commodity'>, 'CASH': <AssetClass.BOND: 'bond'>, 'XOM': <AssetClass.EQUITY: 'equity'>}
Portfolio(name='Diversified Multi-Asset Balanced', assets=10, n_obs=1459, frequency=365)
  weights: {'SPY': 0.2174, 'IWM': 0.1087, 'JPM': 0.1304, 'GLD': 0.1304, 'SLV': 0.087, 'USO': 0.087, 'BTC': 0.1087, 'ETH': 0.0761, 'CASH': 0.163, 'NVDA': -0.1087}
  asset_classes: {'SPY': <AssetClass.EQUITY: 'equity'>, 'IWM': <AssetClass.EQUITY: 'equity'>, 'JPM': <AssetClass.EQUITY: 'equity'>, 'GLD': <AssetClass.COMM

In [154]:
# A shared benchmark for every alpha/beta/capture-ratio style metric below.
benchmark_returns = master["SPY"].pct_change().dropna().rename("SPY")
print(f"Benchmark: SPY, {len(benchmark_returns)} daily returns, {benchmark_returns.index.min().date()} -> {benchmark_returns.index.max().date()}")


Benchmark: SPY, 1458 daily returns, 2022-08-09 -> 2026-08-05


### 4.2 Bonus: `convert_to_base_currency` (FX conversion, self-contained example)


In [155]:
demo_prices = pd.DataFrame({"EUR_STOCK": [100.0, 101.0, 102.5]}, index=pd.date_range("2026-01-01", periods=3))
demo_fx = pd.DataFrame({"EUR": [1.08, 1.09, 1.10]}, index=pd.date_range("2026-01-01", periods=3))
demo_usd = convert_to_base_currency(demo_prices, demo_fx, asset_currencies={"EUR_STOCK": "EUR"}, base_currency="USD")
demo_usd


,EUR_STOCK
2026-01-01,108.0000
2026-01-02,110.0900
2026-01-03,112.7500


## 5. Portfolio Core Accessors & Methods


In [156]:
for label, p in portfolios.items():
    print(f"--- {label} ({p.name}) ---")
    print("asset_names:", p.asset_names)
    print("num_assets:", p.num_assets)
    print("prices.shape:", p.prices.shape)
    print("weights:")
    print(p.weights)
    print("asset_returns().tail(2):")
    print(p.asset_returns().tail(2))
    print("returns().tail(2):")
    print(p.returns().tail(2))
    print("returns(log=True).tail(2):")
    print(p.returns(log=True).tail(2))
    print("returns(period=21).tail(2)  (~monthly blocks):")
    print(p.returns(period=21).tail(2))
    print("price_index().tail(2):")
    print(p.price_index().tail(2))
    print()


--- growth (Growth & Crypto Long/Short) ---
asset_names: ['AAPL', 'MSFT', 'NVDA', 'QQQ', 'BTC', 'ETH', 'GLD', 'CASH', 'XOM']
num_assets: 9
prices.shape: (1459, 9)
weights:
AAPL    0.2444
MSFT    0.2000
NVDA    0.2000
QQQ     0.1333
BTC     0.1556
ETH     0.0889
GLD     0.0889
CASH    0.0556
XOM    -0.1667
Name: weight, dtype: float64
asset_returns().tail(2):
              AAPL    MSFT   NVDA     QQQ    BTC    ETH    GLD   CASH     XOM
timestamp                                                                    
2026-08-04  0.0196  0.0106 0.0256  0.0340 0.0096 0.0050 0.0066 0.0001 -0.0071
2026-08-05 -0.0027 -0.0061 0.0462 -0.0047 0.0107 0.0267 0.0445 0.0001 -0.0115
returns().tail(2):
timestamp
2026-08-04   0.0203
2026-08-05   0.0166
Name: Growth & Crypto Long/Short, dtype: float64
returns(log=True).tail(2):
timestamp
2026-08-04   0.0201
2026-08-05   0.0165
Name: Growth & Crypto Long/Short, dtype: float64
returns(period=21).tail(2)  (~monthly blocks):
2026-07-27   -0.0046
2026-08-05    0

In [157]:
demo_portfolio = build_portfolio("Demo (mutation sandbox)", weights_growth)
print("original weights:", demo_portfolio.weights.round(3).to_dict())
demo_portfolio.set_weights({k: 1.0 for k in demo_portfolio.asset_names})  # unnormalized -> normalize_weights rescales it
print("after set_weights (equal, pre-normalize):", demo_portfolio.weights.round(3).to_dict())

print("original risk_free_rate:", demo_portfolio.risk_free_rate)
demo_portfolio.set_risk_free_rate(0.02)
print("after set_risk_free_rate(0.02):", demo_portfolio.risk_free_rate)
print(repr(demo_portfolio))


original weights: {'AAPL': 0.244, 'MSFT': 0.2, 'NVDA': 0.2, 'QQQ': 0.133, 'BTC': 0.156, 'ETH': 0.089, 'GLD': 0.089, 'CASH': 0.056, 'XOM': -0.167}
after set_weights (equal, pre-normalize): {'AAPL': 0.111, 'MSFT': 0.111, 'NVDA': 0.111, 'QQQ': 0.111, 'BTC': 0.111, 'ETH': 0.111, 'GLD': 0.111, 'CASH': 0.111, 'XOM': 0.111}
original risk_free_rate: 0.0391
after set_risk_free_rate(0.02): 0.02
Portfolio(name='Demo (mutation sandbox)', assets=9, n_obs=1459, frequency=365)


## 6. Metrics -- Returns (`portpy.metrics.returns`)


In [158]:
from portpy.metrics.returns import (
    active_returns,
    annualized_return,
    average_return,
    cagr,
    cumulative_returns,
    excess_returns,
    log_returns,
    prices_from_returns,
    rebased_returns,
    simple_returns,
    total_return,
)

for label, p in portfolios.items():
    sr = simple_returns(p.prices)
    lr = log_returns(p.prices)
    print(f"{label}: simple_returns shape={sr.shape}, log_returns shape={lr.shape}")
    print(sr.iloc[-1].round(4).to_dict())


growth: simple_returns shape=(1458, 9), log_returns shape=(1458, 9)
{'AAPL': -0.0027, 'MSFT': -0.0061, 'NVDA': 0.0462, 'QQQ': -0.0047, 'BTC': 0.0107, 'ETH': 0.0267, 'GLD': 0.0445, 'CASH': 0.0001, 'XOM': -0.0115}
balanced: simple_returns shape=(1458, 10), log_returns shape=(1458, 10)
{'SPY': 0.0003, 'IWM': -0.0037, 'JPM': 0.0084, 'GLD': 0.0445, 'SLV': 0.0449, 'USO': -0.0057, 'BTC': 0.0107, 'ETH': 0.0267, 'CASH': 0.0001, 'NVDA': 0.0462}
macro: simple_returns shape=(1458, 9), log_returns shape=(1458, 9)
{'MSFT': -0.0061, 'QQQ': -0.0047, 'BTC': 0.0107, 'ETH': 0.0267, 'SOL': 0.0118, 'GLD': 0.0445, 'USO': -0.0057, 'SPY': 0.0003, 'CASH': 0.0001}


In [159]:
for label, p in portfolios.items():
    r = p.returns()
    cum = cumulative_returns(r)
    reconstructed_prices = prices_from_returns(r, base=100.0)
    roundtrip_ok = np.allclose(simple_returns(reconstructed_prices).to_numpy(), r.to_numpy())
    print(f"{label}: total cumulative return={cum.iloc[-1]:+.2%}, prices_from_returns roundtrips exactly: {roundtrip_ok}")


growth: total cumulative return=+170.15%, prices_from_returns roundtrips exactly: True
balanced: total cumulative return=+91.29%, prices_from_returns roundtrips exactly: True
macro: total cumulative return=+221.31%, prices_from_returns roundtrips exactly: True


In [160]:
for label, p in portfolios.items():
    rebased = rebased_returns(p.price_index(), base=100.0)
    print(f"{label}: rebased_returns -> starts at {rebased.iloc[0]:.2f}, ends at {rebased.iloc[-1]:.2f}")


growth: rebased_returns -> starts at 100.00, ends at 270.15
balanced: rebased_returns -> starts at 100.00, ends at 191.29
macro: rebased_returns -> starts at 100.00, ends at 321.31


In [161]:
for label, p in portfolios.items():
    tr = p.metrics.total_return()
    tr_result = p.metrics.total_return(as_result=True)
    print(f"{label}: total_return={tr:+.2%}  |  as_result -> {tr_result!r}")
tr_result.explain()


growth: total_return=+170.15%  |  as_result -> total_return=1.70148 % - +170.1% total over the period
balanced: total_return=+91.29%  |  as_result -> total_return=0.912858 % - +91.3% total over the period
macro: total_return=+221.31%  |  as_result -> total_return=2.21313 % - +221.3% total over the period
total_return (metric)

What it is:
  The compounded percentage gain or loss over the entire period, ignoring time - 'if I put money in on day 1 and took it out on the last day, what's my total gain?'

Formula:
  prod(1 + r_t) - 1

How to read it:
  Read directly as a percentage: 0.35 means +35% over the whole period.

Good vs. bad:
  There's no universal 'good' threshold - it depends entirely on the length of the period and the asset class. Compare it to a relevant benchmark over the *same* period rather than judging it in isolation.

Caveats:
  Not annualized, so a 35% return over 1 year and 35% over 10 years look identical here.

This result:
  +221.3% total over the period


"total_return (metric)\n=====================\n\nWhat it is:\n  The compounded percentage gain or loss over the entire period, ignoring time - 'if I put money in on day 1 and took it out on the last day, what's my total gain?'\n\nFormula:\n  prod(1 + r_t) - 1\n\nHow to read it:\n  Read directly as a percentage: 0.35 means +35% over the whole period.\n\nGood vs. bad:\n  There's no universal 'good' threshold - it depends entirely on the length of the period and the asset class. Compare it to a relevant benchmark over the *same* period rather than judging it in isolation.\n\nCaveats:\n  Not annualized, so a 35% return over 1 year and 35% over 10 years look identical here.\n\nThis result:\n  +221.3% total over the period"

In [162]:
for label, p in portfolios.items():
    geo = p.metrics.annualized_return(geometric=True)
    arith = p.metrics.annualized_return(geometric=False)
    print(f"{label}: annualized_return geometric={geo:+.2%}, arithmetic={arith:+.2%}")


growth: annualized_return geometric=+28.25%, arithmetic=+29.13%
balanced: annualized_return geometric=+17.63%, arithmetic=+17.61%
macro: annualized_return geometric=+33.94%, arithmetic=+39.94%


In [163]:
for label, p in portfolios.items():
    c = p.metrics.cagr(as_result=True)
    print(f"{label}: {c!r}")


growth: cagr=0.282473 % - +28.2%/year compounded
balanced: cagr=0.176298 % - +17.6%/year compounded
macro: cagr=0.339387 % - +33.9%/year compounded


In [164]:
for label, p in portfolios.items():
    a = p.metrics.average_return(geometric=False)
    g = p.metrics.average_return(geometric=True)
    print(f"{label}: average_return arithmetic={a:+.4%}, geometric={g:+.4%}")


growth: average_return arithmetic=+0.0798%, geometric=+0.0682%
balanced: average_return arithmetic=+0.0482%, geometric=+0.0445%
macro: average_return arithmetic=+0.1094%, geometric=+0.0801%


In [165]:
for label, p in portfolios.items():
    r = p.returns()
    excess_vs_rate = excess_returns(r, 0.0001)
    excess_vs_bench = excess_returns(r, benchmark_returns)
    active = active_returns(r, benchmark_returns)
    print(
        f"{label}: excess_returns vs flat rate mean={excess_vs_rate.mean():+.5f}, "
        f"excess_returns vs SPY mean={excess_vs_bench.mean():+.5f}, "
        f"active_returns == excess_returns vs benchmark: {active.equals(excess_vs_bench)}"
    )


growth: excess_returns vs flat rate mean=+0.00070, excess_returns vs SPY mean=+0.00033, active_returns == excess_returns vs benchmark: True
balanced: excess_returns vs flat rate mean=+0.00038, excess_returns vs SPY mean=+0.00002, active_returns == excess_returns vs benchmark: True
macro: excess_returns vs flat rate mean=+0.00099, excess_returns vs SPY mean=+0.00063, active_returns == excess_returns vs benchmark: True


## 7. Metrics -- Risk (`portpy.metrics.risk`)


In [166]:
for label, p in portfolios.items():
    var = p.metrics.variance()
    vol_ann = p.metrics.volatility(annualized=True)
    vol_raw = p.metrics.volatility(annualized=False)
    print(f"{label}: variance={var:.6f}, volatility(annualized)={vol_ann:.2%}, volatility(raw)={vol_raw:.4%}")


growth: variance=0.000230, volatility(annualized)=29.00%, volatility(raw)=1.5179%
balanced: variance=0.000075, volatility(annualized)=16.60%, volatility(raw)=0.8689%
macro: variance=0.000727, volatility(annualized)=51.51%, volatility(raw)=2.6959%


In [167]:
for label, p in portfolios.items():
    dd0 = p.metrics.downside_deviation(mar=0.0)
    dd_mar = p.metrics.downside_deviation(mar=0.02)
    sv = p.metrics.semi_variance(mar=0.0)
    print(f"{label}: downside_deviation(mar=0)={dd0:.2%}, downside_deviation(mar=2%)={dd_mar:.2%}, semi_variance={sv:.6f}")


growth: downside_deviation(mar=0)=20.20%, downside_deviation(mar=2%)=20.24%, semi_variance=0.000240
balanced: downside_deviation(mar=0)=10.94%, downside_deviation(mar=2%)=10.99%, semi_variance=0.000071
macro: downside_deviation(mar=0)=22.94%, downside_deviation(mar=2%)=22.99%, semi_variance=0.000298


In [168]:
for label, p in portfolios.items():
    for method in ("historical", "parametric", "cornish_fisher"):
        for confidence in (0.95, 0.99):
            var = p.metrics.value_at_risk(method=method, confidence=confidence)
            print(f"{label}: VaR[{method}, {confidence:.0%}] = {var:+.2%}")
    cvar = p.metrics.conditional_var(confidence=0.95)
    print(f"{label}: CVaR[95%] = {cvar:+.2%}\n")


growth: VaR[historical, 95%] = -2.21%
growth: VaR[historical, 99%] = -3.97%
growth: VaR[parametric, 95%] = -2.42%
growth: VaR[parametric, 99%] = -3.45%
growth: VaR[cornish_fisher, 95%] = -2.05%
growth: VaR[cornish_fisher, 99%] = -11.40%
growth: CVaR[95%] = -3.44%

balanced: VaR[historical, 95%] = -1.26%
balanced: VaR[historical, 99%] = -2.38%
balanced: VaR[parametric, 95%] = -1.38%
balanced: VaR[parametric, 99%] = -1.97%
balanced: VaR[cornish_fisher, 95%] = -0.82%
balanced: VaR[cornish_fisher, 99%] = -4.40%
balanced: CVaR[95%] = -1.93%

macro: VaR[historical, 95%] = -2.65%
macro: VaR[historical, 99%] = -4.48%
macro: VaR[parametric, 95%] = -4.32%
macro: VaR[parametric, 99%] = -6.16%
macro: VaR[cornish_fisher, 95%] = +49.16%
macro: VaR[cornish_fisher, 99%] = +5.98%
macro: CVaR[95%] = -3.89%



In [169]:
for label, p in portfolios.items():
    print(f"{label}: tail_ratio={p.metrics.tail_ratio():.2f}, skewness={p.metrics.skewness():+.2f}, kurtosis={p.metrics.kurtosis():+.2f}")


growth: tail_ratio=1.09, skewness=-0.66, kurtosis=+21.11
balanced: tail_ratio=1.05, skewness=+1.02, kurtosis=+16.84
macro: tail_ratio=1.04, skewness=+16.78, kurtosis=+487.27


In [170]:
for label, p in portfolios.items():
    ui = p.metrics.ulcer_index()
    pi = p.metrics.pain_index()
    print(f"{label}: ulcer_index={ui:.2%}, pain_index={pi:.2%}")


growth: ulcer_index=12.65%, pain_index=9.62%
balanced: ulcer_index=4.69%, pain_index=3.65%
macro: ulcer_index=17.29%, pain_index=12.07%


In [171]:
for label, p in portfolios.items():
    b = p.metrics.beta(benchmark=benchmark_returns)
    te = p.metrics.tracking_error(benchmark=benchmark_returns)
    print(f"{label}: beta vs SPY={b:+.2f}, tracking_error vs SPY={te:.2%}")


growth: beta vs SPY=+1.33, tracking_error vs SPY=20.05%
balanced: beta vs SPY=+0.57, tracking_error vs SPY=15.46%
macro: beta vs SPY=+0.93, tracking_error vs SPY=49.26%


## 8. Metrics -- Performance (`portpy.metrics.performance`)


In [172]:
for label, p in portfolios.items():
    sharpe = p.metrics.sharpe_ratio(as_result=True)
    sortino = p.metrics.sortino_ratio(mar=0.02, as_result=True)
    print(f"{label}: {sharpe!r}  |  {sortino!r}")


growth: sharpe_ratio=0.872115 - sub-par  |  sortino_ratio=1.34106 - good
balanced: sharpe_ratio=0.829783 - sub-par  |  sortino_ratio=1.42272 - good
macro: sharpe_ratio=0.701065 - sub-par  |  sortino_ratio=1.651 - good


In [173]:
for label, p in portfolios.items():
    print(f"{label}: {p.metrics.calmar_ratio(as_result=True)!r}")


growth: calmar_ratio=0.976199 - weak
balanced: calmar_ratio=1.1129 - solid
macro: calmar_ratio=0.744058 - weak


In [174]:
for label, p in portfolios.items():
    o0 = p.metrics.omega_ratio(threshold=0.0)
    o_pos = p.metrics.omega_ratio(threshold=0.05)
    print(f"{label}: omega_ratio(threshold=0)={o0:.2f}, omega_ratio(threshold=5%)={o_pos:.2f}")


growth: omega_ratio(threshold=0)=1.18, omega_ratio(threshold=5%)=1.15
balanced: omega_ratio(threshold=0)=1.18, omega_ratio(threshold=5%)=1.13
macro: omega_ratio(threshold=0)=1.19, omega_ratio(threshold=5%)=1.16


In [175]:
for label, p in portfolios.items():
    ir = p.metrics.information_ratio(benchmark=benchmark_returns, as_result=True)
    print(f"{label}: {ir!r}")


growth: information_ratio=0.606329 - good
balanced: information_ratio=0.0413326 - modest
macro: information_ratio=0.466396 - modest


In [176]:
for label, p in portfolios.items():
    b = p.metrics.beta(benchmark=benchmark_returns)
    tr = p.metrics.treynor_ratio(beta=b, as_result=True)
    print(f"{label}: beta={b:+.2f}  ->  {tr!r}")


growth: beta=+1.33  ->  treynor_ratio=0.190695 - +0.191 excess return per unit of beta
balanced: beta=+0.57  ->  treynor_ratio=0.242222 - +0.242 excess return per unit of beta
macro: beta=+0.93  ->  treynor_ratio=0.389834 - +0.390 excess return per unit of beta


In [177]:
for label, p in portfolios.items():
    m2 = p.metrics.m2_measure(benchmark=benchmark_returns, as_result=True)
    print(f"{label}: {m2!r}")


growth: m2_measure=0.181252 % - +18.1%/yr risk-matched return
balanced: m2_measure=0.174352 % - +17.4%/yr risk-matched return
macro: m2_measure=0.153371 % - +15.3%/yr risk-matched return


In [178]:
for label, p in portfolios.items():
    sterling = p.metrics.sterling_ratio(n=5)
    burke = p.metrics.burke_ratio(n=5)
    print(f"{label}: sterling_ratio(n=5)={sterling:.2f}, burke_ratio(n=5)={burke:.2f}")


growth: sterling_ratio(n=5)=1.10, burke_ratio(n=5)=0.41
balanced: sterling_ratio(n=5)=1.52, burke_ratio(n=5)=0.52
macro: sterling_ratio(n=5)=1.27, burke_ratio(n=5)=0.46


In [179]:
for label, p in portfolios.items():
    print(f"{label}: gain_to_pain_ratio={p.metrics.gain_to_pain_ratio():.2f}")


growth: gain_to_pain_ratio=0.18
balanced: gain_to_pain_ratio=0.18
macro: gain_to_pain_ratio=0.19


In [180]:
for label, p in portfolios.items():
    k3 = p.metrics.kappa_three_ratio(mar=0.0)
    upr = p.metrics.upside_potential_ratio(mar=0.0)
    print(f"{label}: kappa_three_ratio={k3:.2f}, upside_potential_ratio={upr:.2f}")


growth: kappa_three_ratio=0.04, upside_potential_ratio=0.50
balanced: kappa_three_ratio=0.06, upside_potential_ratio=0.55
macro: kappa_three_ratio=0.06, upside_potential_ratio=0.58


## 9. Metrics -- Drawdowns (`portpy.metrics.drawdowns`)


In [181]:
for label, p in portfolios.items():
    dd = p.metrics.drawdown_series()
    mdd = p.metrics.max_drawdown(as_result=True)
    print(f"{label}: drawdown_series range [{dd.min():.2%}, {dd.max():.2%}]  |  {mdd!r}")


growth: drawdown_series range [-28.94%, 0.00%]  |  max_drawdown=-0.28936 % - severe
balanced: drawdown_series range [-15.84%, 0.00%]  |  max_drawdown=-0.158414 % - typical for a risky asset
macro: drawdown_series range [-45.61%, 0.00%]  |  max_drawdown=-0.45613 % - extreme


In [182]:
for label, p in portfolios.items():
    dur = p.metrics.drawdown_duration()
    ttr = p.metrics.time_to_recovery()
    print(f"{label}: current drawdown_duration={int(dur.iloc[-1])} periods, time_to_recovery(worst episode)={ttr}")


growth: current drawdown_duration=281 periods, time_to_recovery(worst episode)=213.0
balanced: current drawdown_duration=86 periods, time_to_recovery(worst episode)=129.0
macro: current drawdown_duration=303 periods, time_to_recovery(worst episode)=None


In [183]:
for label, p in portfolios.items():
    print(f"--- {label} top_n_drawdowns(n=3) ---")
    print(p.metrics.top_n_drawdowns(n=3))


--- growth top_n_drawdowns(n=3) ---
       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2022-08-15 2022-11-09 2023-03-16 -0.2894                  86              213.0000       True
1 2025-10-28 2026-03-30        NaT -0.2880                 153                   NaN      False
2 2024-12-16 2025-04-08 2025-06-25 -0.2843                 113              191.0000       True
--- balanced top_n_drawdowns(n=3) ---
       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2025-01-31 2025-04-08 2025-06-09 -0.1584                  67              129.0000       True
1 2022-08-13 2022-09-27 2023-01-14 -0.1133                  45              154.0000       True
2 2026-01-28 2026-02-05 2026-05-01 -0.1088                   8               93.0000       True
--- macro top_n_drawdowns(n=3) ---
       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2025-10-06 2026-06-10    

In [184]:
for label, p in portfolios.items():
    avg_dd = p.metrics.average_drawdown(as_result=True)
    dar_95 = p.metrics.drawdown_at_risk(confidence=0.95)
    dar_99 = p.metrics.drawdown_at_risk(confidence=0.99)
    print(f"{label}: {avg_dd!r}  |  drawdown_at_risk[95%]={dar_95:.2%}, [99%]={dar_99:.2%}")


growth: average_drawdown=-0.0347243 % - -3.47% typical episode depth  |  drawdown_at_risk[95%]=-24.00%, [99%]=-26.95%
balanced: average_drawdown=-0.0251301 % - -2.51% typical episode depth  |  drawdown_at_risk[95%]=-8.77%, [99%]=-10.83%
macro: average_drawdown=-0.0499834 % - -5.00% typical episode depth  |  drawdown_at_risk[95%]=-39.09%, [99%]=-43.35%


## 10. Metrics -- Rolling & Expanding Windows (`portpy.metrics.rolling`)


In [185]:
from portpy.metrics.risk import skewness as _skewness
from portpy.metrics.rolling import rolling_metric

WINDOW = 60
for label, p in portfolios.items():
    r = p.returns()
    roll_skew = rolling_metric(r, _skewness, window=WINDOW)
    roll_sharpe = p.metrics.rolling_sharpe(window=WINDOW)
    roll_vol = p.metrics.rolling_volatility(window=WINDOW)
    print(
        f"{label}: rolling_metric(skewness, w={WINDOW}) last={roll_skew.dropna().iloc[-1]:+.2f}, "
        f"rolling_sharpe last={roll_sharpe.dropna().iloc[-1]:+.2f}, "
        f"rolling_volatility last={roll_vol.dropna().iloc[-1]:.2%}"
    )


growth: rolling_metric(skewness, w=60) last=+0.21, rolling_sharpe last=+1.90, rolling_volatility last=26.30%
balanced: rolling_metric(skewness, w=60) last=-0.14, rolling_sharpe last=+1.23, rolling_volatility last=13.86%
macro: rolling_metric(skewness, w=60) last=+0.17, rolling_sharpe last=+2.26, rolling_volatility last=29.27%


In [186]:
from portpy.metrics.rolling import rolling_correlation

for label, p in portfolios.items():
    rb = p.metrics.rolling_beta(benchmark=benchmark_returns, window=WINDOW)
    print(f"{label}: rolling_beta(w={WINDOW}) last={rb.dropna().iloc[-1]:+.2f}")

rc = rolling_correlation(p_growth.returns(), p_macro.returns(), window=WINDOW)
print(f"rolling_correlation(growth vs macro, w={WINDOW}) last={rc.dropna().iloc[-1]:+.2f}")


growth: rolling_beta(w=60) last=+1.27
balanced: rolling_beta(w=60) last=+0.43
macro: rolling_beta(w=60) last=+1.09
rolling_correlation(growth vs macro, w=60) last=+0.85


In [187]:
from portpy.metrics.performance import sharpe_ratio as _sharpe_ratio
from portpy.metrics.rolling import expanding_metric

for label, p in portfolios.items():
    exp_sharpe = expanding_metric(p.returns(), _sharpe_ratio, min_periods=30)
    print(f"{label}: expanding_metric(sharpe_ratio) last value={exp_sharpe.dropna().iloc[-1]:+.2f}")


growth: expanding_metric(sharpe_ratio) last value=+0.83
balanced: expanding_metric(sharpe_ratio) last value=+0.88
macro: expanding_metric(sharpe_ratio) last value=+0.64


## 11. Metrics -- Distributions (`portpy.metrics.distributions`)


In [188]:
for label, p in portfolios.items():
    print(f"--- {label} describe() ---")
    print(p.metrics.describe())


--- growth describe() ---
count      1,458.0000
mean           0.0008
std            0.0152
min           -0.1831
25%           -0.0053
50%            0.0005
75%            0.0072
max            0.1309
skew          -0.6641
kurtosis      21.1097
dtype: float64
--- balanced describe() ---
count      1,458.0000
mean           0.0005
std            0.0087
min           -0.0512
25%           -0.0033
50%            0.0004
75%            0.0044
max            0.1031
skew           1.0159
kurtosis      16.8379
dtype: float64
--- macro describe() ---
count      1,458.0000
mean           0.0011
std            0.0270
min           -0.1221
25%           -0.0080
50%            0.0003
75%            0.0096
max            0.7835
skew          16.7768
kurtosis     487.2715
dtype: float64


In [189]:
for label, p in portfolios.items():
    nt = p.metrics.normality_test()
    print(f"{label}: {nt}")


growth: {'statistic': 26982.524186054383, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}
balanced: {'statistic': 17347.562888129647, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}
macro: {'statistic': 14393429.728634782, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}


In [190]:
for label, p in portfolios.items():
    bw = p.metrics.best_worst_periods(n=3)
    print(f"--- {label} ---")
    print("best:")
    print(bw["best"])
    print("worst:")
    print(bw["worst"])


--- growth ---
best:
timestamp
2025-04-09   0.1309
2022-11-10   0.1091
2023-05-25   0.0646
Name: Growth & Crypto Long/Short, dtype: float64
worst:
timestamp
2024-06-10   -0.1831
2022-09-13   -0.0716
2022-08-26   -0.0592
Name: Growth & Crypto Long/Short, dtype: float64
--- balanced ---
best:
timestamp
2024-06-10   0.1031
2025-04-09   0.0550
2022-11-10   0.0408
Name: Diversified Multi-Asset Balanced, dtype: float64
worst:
timestamp
2026-02-05   -0.0512
2026-01-30   -0.0430
2022-11-09   -0.0374
Name: Diversified Multi-Asset Balanced, dtype: float64
--- macro ---
best:
timestamp
2024-08-26   0.7835
2022-11-10   0.1095
2025-03-02   0.0811
Name: Long/Short Macro, dtype: float64
worst:
timestamp
2022-11-09   -0.1221
2026-02-05   -0.0940
2025-03-03   -0.0735
Name: Long/Short Macro, dtype: float64


In [191]:
for label, p in portfolios.items():
    wr = p.metrics.win_rate()
    wlr = p.metrics.win_loss_ratio()
    ppp = p.metrics.positive_periods_pct()
    print(f"{label}: win_rate={wr:.1%}, win_loss_ratio={wlr:.2f}, positive_periods_pct={ppp:.1%} (matches win_rate: {np.isclose(wr, ppp)})")


growth: win_rate=53.4%, win_loss_ratio=1.03, positive_periods_pct=53.4% (matches win_rate: True)
balanced: win_rate=54.0%, win_loss_ratio=1.01, positive_periods_pct=54.0% (matches win_rate: True)
macro: win_rate=51.6%, win_loss_ratio=1.11, positive_periods_pct=51.6% (matches win_rate: True)


In [192]:
for label, p in portfolios.items():
    print(f"--- {label} monthly_returns_table() (tail) ---")
    print(p.metrics.monthly_returns_table().tail(4))


--- growth monthly_returns_table() (tail) ---
         Jan     Feb     Mar     Apr    May     Jun     Jul     Aug     Sep    Oct     Nov     Dec    Year
year                                                                                                      
2023  0.1981  0.0508  0.1715  0.0173 0.1200  0.0755  0.0185 -0.0402 -0.0654 0.0721  0.1318  0.0567  1.1170
2024  0.0456  0.1648  0.0474 -0.0676 0.1527 -0.1352 -0.0094 -0.0160  0.0402 0.0173  0.1274  0.0089  0.3912
2025 -0.0101 -0.0626 -0.0864  0.0538 0.1381  0.0571  0.0906  0.0230  0.0672 0.0290 -0.0755 -0.0081  0.2076
2026 -0.0721 -0.0732 -0.0479  0.1298 0.0751 -0.1148  0.0638  0.0536     NaN    NaN     NaN     NaN -0.0132
--- balanced monthly_returns_table() (tail) ---
         Jan     Feb     Mar     Apr     May     Jun    Jul     Aug     Sep     Oct     Nov     Dec   Year
year                                                                                                      
2023  0.0645 -0.0414  0.0279  0.0229 -0.0598  0.03

In [193]:
for label, p in portfolios.items():
    counts, edges = p.metrics.return_histogram_data(bins=20)
    print(f"{label}: return_histogram_data -> {len(counts)} bins, busiest bin count={counts.max()}, range=[{edges[0]:+.3%}, {edges[-1]:+.3%}]")


growth: return_histogram_data -> 20 bins, busiest bin count=814, range=[-18.305%, +13.094%]
balanced: return_histogram_data -> 20 bins, busiest bin count=691, range=[-5.119%, +10.310%]
macro: return_histogram_data -> 20 bins, busiest bin count=1171, range=[-12.213%, +78.347%]


## 12. Metrics -- Benchmark-Relative (`portpy.metrics.benchmarks`)


In [194]:
for label, p in portfolios.items():
    a = p.metrics.alpha(benchmark=benchmark_returns, as_result=True)
    corr = p.metrics.correlation(benchmark=benchmark_returns)
    r2 = p.metrics.r_squared(benchmark=benchmark_returns)
    print(f"{label}: {a!r}  |  correlation={corr:+.2f}  |  r_squared={r2:.1%}")


growth: alpha=0.0818766 % - +8.2%/yr (outperforming what beta alone predicts)  |  correlation=+0.75  |  r_squared=55.6%
balanced: alpha=0.0650719 % - +6.5%/yr (outperforming what beta alone predicts)  |  correlation=+0.56  |  r_squared=31.2%
macro: alpha=0.270414 % - +27.0%/yr (outperforming what beta alone predicts)  |  correlation=+0.29  |  r_squared=8.6%


In [195]:
for label, p in portfolios.items():
    up = p.metrics.up_capture_ratio(benchmark=benchmark_returns)
    down = p.metrics.down_capture_ratio(benchmark=benchmark_returns)
    overall = p.metrics.capture_ratio(benchmark=benchmark_returns)
    print(f"{label}: up_capture={up:.2f}, down_capture={down:.2f}, capture_ratio={overall:.2f}")


growth: up_capture=2.50, down_capture=1.04, capture_ratio=1.67
balanced: up_capture=0.30, down_capture=0.82, capture_ratio=1.04
macro: up_capture=0.84, down_capture=0.96, capture_ratio=2.00


In [196]:
for label, p in portfolios.items():
    ba = p.metrics.batting_average(benchmark=benchmark_returns)
    print(f"{label}: batting_average={ba:.1%}")


growth: batting_average=52.7%
balanced: batting_average=50.1%
macro: batting_average=49.3%


## 13. Metrics -- Regressions (`portpy.metrics.regressions`)


In [197]:
from portpy.metrics.regressions import regression_summary

for label, p in portfolios.items():
    model = p.metrics.linear_regression(x=benchmark_returns)
    summary = regression_summary(model)
    print(f"--- {label}: single-factor (vs SPY) ---")
    print(summary)
    print(f"R^2={summary.attrs['r_squared']:.3f}, adj R^2={summary.attrs['adj_r_squared']:.3f}\n")


--- growth: single-factor (vs SPY) ---
        coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0002   0.0003  0.6831   0.4946   -0.0003     0.0007
SPY   1.3263   0.0311 42.6725   0.0000    1.2653     1.3872
R^2=0.556, adj R^2=0.555

--- balanced: single-factor (vs SPY) ---
        coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0002   0.0002  1.1531   0.2491   -0.0002     0.0006
SPY   0.5687   0.0221 25.6832   0.0000    0.5252     0.6121
R^2=0.312, adj R^2=0.311

--- macro: single-factor (vs SPY) ---
        coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0007   0.0007  0.9814   0.3265   -0.0007     0.0020
SPY   0.9263   0.0792 11.6989   0.0000    0.7709     1.0816
R^2=0.086, adj R^2=0.085



In [198]:
gld_returns = master["GLD"].pct_change().dropna().rename("GLD")
factors = pd.concat([benchmark_returns, gld_returns], axis=1)

for label, p in portfolios.items():
    model = p.metrics.linear_regression(x=factors)
    summary = regression_summary(model)
    print(f"--- {label}: two-factor (SPY, GLD) ---")
    print(summary)


--- growth: two-factor (SPY, GLD) ---
        coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0001   0.0003  0.5008   0.6166   -0.0004     0.0007
SPY   1.3046   0.0316 41.3178   0.0000    1.2426     1.3665
GLD   0.0923   0.0264  3.5019   0.0005    0.0406     0.1441
--- balanced: two-factor (SPY, GLD) ---
        coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0001   0.0002  0.3054   0.7601   -0.0003     0.0004
SPY   0.4948   0.0203 24.3550   0.0000    0.4549     0.5346
GLD   0.3146   0.0170 18.5438   0.0000    0.2813     0.3479
--- macro: two-factor (SPY, GLD) ---
        coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0005   0.0007  0.7573   0.4490   -0.0008     0.0018
SPY   0.8577   0.0802 10.6879   0.0000    0.7003     1.0151
GLD   0.2918   0.0670  4.3548   0.0000    0.1604     0.4233


In [199]:
for label, p in portfolios.items():
    rolling_coefs = p.metrics.rolling_regression(x=benchmark_returns, window=WINDOW)
    print(f"--- {label} rolling_regression(window={WINDOW}) tail ---")
    print(rolling_coefs.tail(3))


--- growth rolling_regression(window=60) tail ---
            const    SPY
timestamp               
2026-08-03 0.0000 1.3642
2026-08-04 0.0002 1.2732
2026-08-05 0.0005 1.2706
--- balanced rolling_regression(window=60) tail ---
             const    SPY
timestamp                
2026-08-03 -0.0002 0.5636
2026-08-04  0.0001 0.4295
2026-08-05  0.0002 0.4281
--- macro rolling_regression(window=60) tail ---
            const    SPY
timestamp               
2026-08-03 0.0004 1.2751
2026-08-04 0.0007 1.0977
2026-08-05 0.0011 1.0942


## 14. Metrics -- Covariance & Risk Decomposition (`portpy.metrics.covariance`)


In [200]:
for label, p in portfolios.items():
    corr = p.metrics.correlation_matrix()
    print(f"--- {label} correlation_matrix ---")
    print(corr.round(2))


--- growth correlation_matrix ---
       AAPL   MSFT   NVDA    QQQ    BTC    ETH    GLD    CASH     XOM
AAPL 1.0000 0.4600 0.3200 0.6600 0.2000 0.2300 0.0900  0.0200  0.1800
MSFT 0.4600 1.0000 0.3700 0.6700 0.2500 0.2600 0.1000  0.0400  0.0400
NVDA 0.3200 0.3700 1.0000 0.5500 0.1900 0.2100 0.0800  0.0200  0.0300
QQQ  0.6600 0.6700 0.5500 1.0000 0.3500 0.3800 0.2000  0.0300  0.1200
BTC  0.2000 0.2500 0.1900 0.3500 1.0000 0.8200 0.1400  0.0600  0.0800
ETH  0.2300 0.2600 0.2100 0.3800 0.8200 1.0000 0.1300  0.0300  0.1000
GLD  0.0900 0.1000 0.0800 0.2000 0.1400 0.1300 1.0000  0.0100  0.0800
CASH 0.0200 0.0400 0.0200 0.0300 0.0600 0.0300 0.0100  1.0000 -0.0300
XOM  0.1800 0.0400 0.0300 0.1200 0.0800 0.1000 0.0800 -0.0300  1.0000
--- balanced correlation_matrix ---
        SPY    IWM    JPM    GLD     SLV     USO    BTC    ETH    CASH   NVDA
SPY  1.0000 0.8400 0.6100 0.2000  0.2900  0.0700 0.3600 0.3900  0.0300 0.4900
IWM  0.8400 1.0000 0.6000 0.2100  0.2800  0.0500 0.3800 0.3900  0.0100 0.3

In [201]:
for label, p in portfolios.items():
    pv = p.metrics.portfolio_variance(as_result=True)
    pvol = p.metrics.portfolio_volatility(as_result=True)
    print(f"{label}: {pv!r}  |  {pvol!r}")


growth: portfolio_variance=0.000230405  |  portfolio_volatility=0.0151791 - 1.52%
balanced: portfolio_variance=7.54928e-05  |  portfolio_volatility=0.00868866 - 0.87%
macro: portfolio_variance=0.000726789  |  portfolio_volatility=0.026959 - 2.70%


In [202]:
for label, p in portfolios.items():
    dr = p.metrics.diversification_ratio(as_result=True)
    print(f"{label}: {dr!r}")


growth: diversification_ratio=1.35696 - 1.36x - diversification is reducing risk to about 74% of the undiversified level
balanced: diversification_ratio=1.26198 - 1.26x - diversification is reducing risk to about 79% of the undiversified level
macro: diversification_ratio=1.35705 - 1.36x - diversification is reducing risk to about 74% of the undiversified level


In [203]:
for label, p in portfolios.items():
    mctr = p.metrics.marginal_contribution_to_risk()
    cctr = p.metrics.component_contribution_to_risk()
    breakdown = pd.DataFrame({"weight": p.weights, "MCTR": mctr, "CCTR": cctr, "CCTR_pct": cctr / cctr.sum()})
    print(f"--- {label} risk decomposition ---")
    print(breakdown.round(4))


--- growth risk decomposition ---
      weight    MCTR   CCTR  CCTR_pct
AAPL  0.2444  0.0087 0.0021    0.1406
MSFT  0.2000  0.0094 0.0019    0.1235
NVDA  0.2000  0.0265 0.0053    0.3497
QQQ   0.1333  0.0090 0.0012    0.0787
BTC   0.1556  0.0159 0.0025    0.1632
ETH   0.0889  0.0220 0.0020    0.1286
GLD   0.0889  0.0021 0.0002    0.0123
CASH  0.0556  0.0000 0.0000    0.0000
XOM  -0.1667 -0.0003 0.0001    0.0033
--- balanced risk decomposition ---
      weight    MCTR   CCTR  CCTR_pct
SPY   0.2174  0.0048 0.0010    0.1192
IWM   0.1087  0.0071 0.0008    0.0884
JPM   0.1304  0.0063 0.0008    0.0951
GLD   0.1304  0.0048 0.0006    0.0713
SLV   0.0870  0.0109 0.0010    0.1094
USO   0.0870  0.0047 0.0004    0.0467
BTC   0.1087  0.0178 0.0019    0.2225
ETH   0.0761  0.0243 0.0018    0.2124
CASH  0.1630  0.0000 0.0000    0.0000
NVDA -0.1087 -0.0028 0.0003    0.0350
--- macro risk decomposition ---
      weight    MCTR    CCTR  CCTR_pct
MSFT  0.1176  0.0034  0.0004    0.0148
QQQ   0.1765  0.0033 

In [204]:
for label, p in portfolios.items():
    dr_annualized_cov = p.metrics.diversification_ratio(cov_matrix=p.metrics.covariance_matrix(annualized=True))
    print(f"{label}: diversification_ratio using an explicit annualized cov_matrix override = {dr_annualized_cov:.2f}")


growth: diversification_ratio using an explicit annualized cov_matrix override = 1.36
balanced: diversification_ratio using an explicit annualized cov_matrix override = 1.26
macro: diversification_ratio using an explicit annualized cov_matrix override = 1.36


## 15. Metrics -- One-Shot Summaries (`portpy.metrics.summary`)


In [205]:
for label, p in portfolios.items():
    print("=" * 20, label.upper(), "TEARSHEET", "=" * 20)
    for metric_name, result in p.metrics.tearsheet_summary().items():
        print(f"  {metric_name:22s} {float(result):>10.4f}   ({result.interpretation})")
    print()


==================== GROWTH TEARSHEET ====================
  total_return               1.7015   (+170.1% total over the period)
  annualized_return          0.2825   (+28.2% per year)
  cagr                       0.2825   (+28.2%/year compounded)
  volatility                 0.2900   (29.0%/yr (high - equity-like or more volatile))
  sharpe_ratio               0.8721   (sub-par)
  sortino_ratio              1.4420   (good)
  calmar_ratio               0.9762   (weak)
  max_drawdown              -0.2894   (severe)
  value_at_risk_95          -0.0221   (-2.21% - expect a worse-than-this loss only in the excluded tail probability)
  conditional_var_95        -0.0344   (-3.44% average loss in the worst-case tail)
  skewness                  -0.6641   (-0.66 (negative skew - watch for rare large losses))
  kurtosis                  21.1097   (+21.11 (fat tails - Normal-based risk estimates will understate real risk))
  win_rate                   0.5343   (53.4% of periods were positive)

=

In [206]:
for label, p in portfolios.items():
    print(f"--- {label} vs SPY ---")
    print(p.metrics.compare_to_benchmark(benchmark=benchmark_returns))
    print()


--- growth vs SPY ---
                    portfolio  benchmark  difference
annualized_return      0.2825     0.1694      0.1131
volatility             0.2900     0.1630      0.1270
sharpe_ratio           0.8721     0.8058      0.0663
max_drawdown          -0.2894    -0.1900     -0.0994
beta                   1.3263        NaN         NaN
alpha                  0.0819        NaN         NaN
correlation            0.7454        NaN         NaN
information_ratio      0.6063        NaN         NaN
up_capture_ratio       2.5033        NaN         NaN
down_capture_ratio     1.0421        NaN         NaN
batting_average        0.5274        NaN         NaN

--- balanced vs SPY ---
                    portfolio  benchmark  difference
annualized_return      0.1763     0.1694      0.0069
volatility             0.1660     0.1630      0.0030
sharpe_ratio           0.8298     0.8058      0.0239
max_drawdown          -0.1584    -0.1900      0.0316
beta                   0.5687        NaN         NaN

## 16. Metrics -- Transaction Costs (`portpy.metrics.costs`)


In [207]:
from portpy.metrics.costs import net_of_costs_returns, turnover_from_weights
from portpy.metrics.returns import total_return as _total_return


def simulate_weight_drift(returns_df: pd.DataFrame, target_weights: pd.Series, rebalance_every: int = 21) -> pd.DataFrame:
    # Buy-and-hold weight drift between periodic rebalances back to target - a stand-in for a real trade log.
    aligned = returns_df[target_weights.index]
    history = []
    current = target_weights.copy()
    for i, date in enumerate(aligned.index):
        if i > 0 and i % rebalance_every == 0:
            current = target_weights.copy()
        history.append(current.copy())
        port_r = float((current * aligned.loc[date]).sum())
        current = current * (1.0 + aligned.loc[date]) / (1.0 + port_r)
    return pd.DataFrame(history, index=aligned.index)


for label, p in portfolios.items():
    weights_history = simulate_weight_drift(p.asset_returns(), p.weights, rebalance_every=21)
    turnover = turnover_from_weights(weights_history)
    print(f"{label}: turnover_from_weights -> mean={turnover.mean():.2%}, max={turnover.max():.2%} (rebalance days spike, drift days near 0)")


growth: turnover_from_weights -> mean=0.87%, max=66.67% (rebalance days spike, drift days near 0)
balanced: turnover_from_weights -> mean=0.71%, max=60.87% (rebalance days spike, drift days near 0)
macro: turnover_from_weights -> mean=1.26%, max=79.41% (rebalance days spike, drift days near 0)


In [212]:
for label, p in portfolios.items():
    weights_history = simulate_weight_drift(p.asset_returns(), p.weights, rebalance_every=21)
    print(weights_history.tail(5))
    turnover = turnover_from_weights(weights_history)
    net_variable = net_of_costs_returns(p.returns(), turnover, cost_bps=10)
    net_constant = net_of_costs_returns(p.returns(), 0.05, cost_bps=10)
    gross_total = _total_return(p.returns())
    net_variable_total = _total_return(net_variable)
    net_constant_total = _total_return(net_constant)
    print(
        f"{label}: gross total_return={gross_total:+.2%}, "
        f"net (variable turnover)={net_variable_total:+.2%}, "
        f"net (constant 5% turnover)={net_constant_total:+.2%}"
    )

    portpy.explain("turnover_from_weights")

             AAPL   MSFT   NVDA    QQQ    BTC    ETH    GLD   CASH     XOM
timestamp                                                                 
2026-08-01 0.2200 0.2344 0.2005 0.1320 0.1505 0.0858 0.0865 0.0545 -0.1643
2026-08-02 0.2201 0.2346 0.2007 0.1321 0.1505 0.0851 0.0866 0.0546 -0.1644
2026-08-03 0.2193 0.2338 0.2000 0.1316 0.1518 0.0867 0.0863 0.0544 -0.1638
2026-08-04 0.2123 0.2417 0.2028 0.1319 0.1494 0.0843 0.0850 0.0536 -0.1610
2026-08-05 0.2122 0.2395 0.2039 0.1338 0.1479 0.0830 0.0839 0.0526 -0.1567
growth: gross total_return=+170.15%, net (variable turnover)=+166.75%, net (constant 5% turnover)=+151.16%
turnover_from_weights (metric)

What it is:
  How much of the portfolio was bought/sold each period, as a fraction of its value - the raw input needed to estimate transaction costs.

Formula:
  sum(|w_t - w_(t-1)|) / 2

How to read it:
  A turnover of 0.20 means 20% of the portfolio's value was traded that period (e.g. selling 10% of one asset and buying 10% of anot

## 17. The Explainability Layer (`portpy.explain`)


In [213]:
print("Registered explanation names by category:")
print(" metric:", len(available("metric")))
print(" chart: ", len(available("chart")))

portpy.explain("sharpe_ratio")


Registered explanation names by category:
 metric: 56
 chart:  6
sharpe_ratio (metric)

What it is:
  The most widely used risk-adjusted return measure: excess return earned per unit of total volatility taken on.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means you earned, on average, one standard deviation of excess return for the volatility you took on.

Good vs. bad:
  Rules of thumb: <0 poor (lost money net of the risk-free rate), 0-1 sub-par, 1-2 good, 2-3 very good, >3 excellent (and worth double-checking for overfitting or a very short sample).

Caveats:
  Assumes returns are roughly symmetric - it penalizes upside volatility just as much as downside, and can be misleadingly high for strategies with rare, large negative tail events (e.g. option-selling). Pair with sortino_ratio and max_drawdown.


'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  The most widely used risk-adjusted return measure: excess return earned per unit of total volatility taken on.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means you earned, on average, one standard deviation of excess return for the volatility you took on.\n\nGood vs. bad:\n  Rules of thumb: <0 poor (lost money net of the risk-free rate), 0-1 sub-par, 1-2 good, 2-3 very good, >3 excellent (and worth double-checking for overfitting or a very short sample).\n\nCaveats:\n  Assumes returns are roughly symmetric - it penalizes upside volatility just as much as downside, and can be misleadingly high for strategies with rare, large negative tail events (e.g. option-selling). Pair with sortino_ratio and max_drawdown.'

In [214]:
sharpe_result = p_growth.metrics.sharpe_ratio(as_result=True)
print("isinstance of float:", isinstance(sharpe_result, float))
print("str():", str(sharpe_result))
print("repr():", repr(sharpe_result))
print("arithmetic still works: sharpe_result * 2 =", sharpe_result * 2)
print("interpretation:", sharpe_result.interpretation)
sharpe_result.explain()


isinstance of float: True
str(): 0.8721152104041359
repr(): sharpe_ratio=0.872115 - sub-par
arithmetic still works: sharpe_result * 2 = 1.7442304208082717
interpretation: sub-par
sharpe_ratio (metric)

What it is:
  The most widely used risk-adjusted return measure: excess return earned per unit of total volatility taken on.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means you earned, on average, one standard deviation of excess return for the volatility you took on.

Good vs. bad:
  Rules of thumb: <0 poor (lost money net of the risk-free rate), 0-1 sub-par, 1-2 good, 2-3 very good, >3 excellent (and worth double-checking for overfitting or a very short sample).

Caveats:
  Assumes returns are roughly symmetric - it penalizes upside volatility just as much as downside, and can be misleadingly high for strategies with rare, large negative tail events (e.g. option-selling). Pair with sortino_ratio and max_drawdown.

This re

'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  The most widely used risk-adjusted return measure: excess return earned per unit of total volatility taken on.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means you earned, on average, one standard deviation of excess return for the volatility you took on.\n\nGood vs. bad:\n  Rules of thumb: <0 poor (lost money net of the risk-free rate), 0-1 sub-par, 1-2 good, 2-3 very good, >3 excellent (and worth double-checking for overfitting or a very short sample).\n\nCaveats:\n  Assumes returns are roughly symmetric - it penalizes upside volatility just as much as downside, and can be misleadingly high for strategies with rare, large negative tail events (e.g. option-selling). Pair with sortino_ratio and max_drawdown.\n\nThis result:\n  sub-par'

In [215]:
comparison = p_growth.metrics.compare_to_benchmark(benchmark=benchmark_returns)
portpy.explain(comparison)  # dispatches via comparison.attrs["portpy_explanation"]

portpy.explain(sharpe_result)  # dispatches via MetricResult.name, auto-injecting its own value


compare_to_benchmark (metric)

What it is:
  A side-by-side table of the portfolio vs. a benchmark: absolute metrics for both (return, volatility, Sharpe, max drawdown) plus relative metrics that only make sense for the portfolio (beta, alpha, correlation, capture ratios).

How to read it:
  The 'difference' column is `portfolio - benchmark` where both exist; it's NaN for relative-only rows (beta, alpha, etc.) since there's no benchmark-side equivalent to subtract.

Good vs. bad:
  A positive difference on return/Sharpe and a less-negative difference on max_drawdown are the 'beating the benchmark, more safely' pattern to look for.
sharpe_ratio (metric)

What it is:
  The most widely used risk-adjusted return measure: excess return earned per unit of total volatility taken on.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means you earned, on average, one standard deviation of excess return for the volatility you took on.

Goo

'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  The most widely used risk-adjusted return measure: excess return earned per unit of total volatility taken on.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means you earned, on average, one standard deviation of excess return for the volatility you took on.\n\nGood vs. bad:\n  Rules of thumb: <0 poor (lost money net of the risk-free rate), 0-1 sub-par, 1-2 good, 2-3 very good, >3 excellent (and worth double-checking for overfitting or a very short sample).\n\nCaveats:\n  Assumes returns are roughly symmetric - it penalizes upside volatility just as much as downside, and can be misleadingly high for strategies with rare, large negative tail events (e.g. option-selling). Pair with sortino_ratio and max_drawdown.\n\nThis result:\n  sub-par'

In [216]:
card = get("max_drawdown")
print(card.name, "|", card.category)
print(card.summary)
print(card.formula)


max_drawdown | metric
The single worst peak-to-trough loss the portfolio ever experienced.
min(drawdown_series)


## 18. `Portfolio.metrics` Auto-Fill Mechanics

`portfolio.metrics.<fn>()` is a thin, dynamically-bound wrapper around the plain function -
useful for debugging exactly what it does and doesn't auto-supply for you.


In [217]:
from portpy.metrics.risk import beta as _beta
from portpy.metrics.risk import volatility as _volatility

p = p_balanced

# 1. No args: every eligible parameter (`returns`, `periods_per_year`, ...) is auto-filled from the portfolio.
auto_vol = p.metrics.volatility()
manual_vol = _volatility(p.returns(), periods_per_year=p.frequency)
print("volatility(): auto-fill matches manual call:", np.isclose(auto_vol, manual_vol))

# 2. kwargs still trigger auto-fill for every OTHER param - only the one you name is overridden.
override_vol = p.metrics.volatility(periods_per_year=252)
print(f"volatility(periods_per_year=252) -> {override_vol:.2%}  (portfolio frequency is {p.frequency})")

# 3. `benchmark` isn't one of the auto-filled names (returns/y/prices/weights/cov_matrix/rf/periods_per_year),
#    so it must always be supplied explicitly - auto-fill only ever covers the portfolio's OWN data.
auto_beta = p.metrics.beta(benchmark=benchmark_returns)
manual_beta = _beta(p.returns(), benchmark_returns)
print("beta(benchmark=...): auto-fill matches manual call:", np.isclose(auto_beta, manual_beta))

# 4. Any positional argument disables auto-fill for the WHOLE call - the guard the docstring warns about.
positional_vol = p.metrics.volatility(p.returns(), True)  # (returns, annualized) positionally
print(
    f"volatility(returns, annualized=True) positionally -> {positional_vol:.4%} "
    f"(periods_per_year silently reverts to the function's own default of 252, not p.frequency={p.frequency})"
)


volatility(): auto-fill matches manual call: True
volatility(periods_per_year=252) -> 13.79%  (portfolio frequency is 365)
beta(benchmark=...): auto-fill matches manual call: True
volatility(returns, annualized=True) positionally -> 13.7928% (periods_per_year silently reverts to the function's own default of 252, not p.frequency=365)
